In [1]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from tqdm import tqdm


In [2]:
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved()  / 1024**2:.2f} MB")
print(f"CUDA Available: {torch.cuda.is_available()}")

GPU Memory Allocated: 0.00 MB
GPU Memory Reserved:  0.00 MB
CUDA Available: True


In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [5]:
"""Dataset paths"""

TRAIN_REAL   = "/kaggle/input/anti-spoofing-replay-attack-frames/FULL_DATASET_FRAMES/train/real"
TRAIN_ATTACK = "/kaggle/input/anti-spoofing-replay-attack-frames/FULL_DATASET_FRAMES/train/attack"
TEST_REAL    = "/kaggle/input/anti-spoofing-replay-attack-frames/FULL_DATASET_FRAMES/test/real"
TEST_ATTACK  = "/kaggle/input/anti-spoofing-replay-attack-frames/FULL_DATASET_FRAMES/test/attack"


In [6]:
"""Dataset class"""

class AntispoofDataset(Dataset):
    def __init__(self, real_path: str, attack_path: str, transform=None):
        self.samples   = []
        self.transform = transform

        for root_path, label in [(real_path, 1), (attack_path, 0)]:
            for identity in os.listdir(root_path):
                identity_dir = os.path.join(root_path, identity)
                if not os.path.isdir(identity_dir):
                    continue
                for img_name in os.listdir(identity_dir):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.samples.append((os.path.join(identity_dir, img_name), label))

        real_count   = sum(1 for _, l in self.samples if l == 1)
        attack_count = sum(1 for _, l in self.samples if l == 0)
        print(f"Loaded {len(self.samples)} samples — Real: {real_count} | Attack: {attack_count}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)


In [7]:
"""Data augmentation"""

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.3),
    transforms.ToTensor(),
    # Gaussian noise added after ToTensor so it operates on float tensors
    transforms.RandomApply(
        [transforms.Lambda(lambda x: (x + 0.01 * torch.randn_like(x)).clamp(0, 1))], p=0.3
    ),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


In [8]:
"""Datasets and loaders"""

train_dataset = AntispoofDataset(TRAIN_REAL, TRAIN_ATTACK, train_transform)
test_dataset  = AntispoofDataset(TEST_REAL,  TEST_ATTACK,  test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)


Loaded 2575 samples — Real: 1216 | Attack: 1359
Loaded 1302 samples — Real: 637 | Attack: 665


In [9]:
class DepthwiseBlock(nn.Module):
    """Depthwise-separable residual block with correct BN/ReLU placement."""
    def __init__(self, ch: int):
        super().__init__()
        self.block = nn.Sequential(
            # Depthwise
            nn.Conv2d(ch, ch, 3, padding=1, groups=ch, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
            # Pointwise
            nn.Conv2d(ch, ch, 1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)

In [10]:
"""CNN — Texture Branch V2 + DepthwiseBlock"""

class MobileNetTexture(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = timm.create_model(
            "mobilenetv3_small_100",
            pretrained=True,
            features_only=True,
        )
        self.backbone = backbone

        feature_info   = backbone.feature_info
        mid_channels   = feature_info[2]["num_chs"]
        final_channels = feature_info[-1]["num_chs"]

        self.texture_branch = nn.Sequential(
            nn.Conv2d(mid_channels, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            DepthwiseBlock(64),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(final_channels + 64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)

        mid_feat   = features[2]
        final_feat = features[-1]

        texture_feat = self.texture_branch(mid_feat)

        main_feat = self.global_pool(final_feat)
        main_feat = main_feat.view(main_feat.size(0), -1)

        combined = torch.cat([main_feat, texture_feat], dim=1)
        return self.classifier(combined)


In [11]:
"""Model setup"""

model     = MobileNetTexture().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

NUM_EPOCHS = 20
best_acc   = 0.0
CKPT_PATH  = "/kaggle/working/model.pth"


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [12]:
"""Training loop"""
for epoch in range(NUM_EPOCHS):

    # Train 
    model.train()
    train_correct = 0
    train_total   = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images).squeeze(1)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds          = (torch.sigmoid(outputs) > 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

    train_acc = 100.0 * train_correct / train_total

    # Eval 
    model.eval()
    test_correct = 0
    test_total   = 0
    all_probs    = []
    all_labels   = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [eval] "):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images).squeeze(1)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            test_correct += (preds == labels).sum().item()
            test_total   += labels.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_acc = 100.0 * test_correct / test_total
    auc      = roc_auc_score(all_labels, all_probs)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Acc : {train_acc:.2f}%")
    print(f"  Test  Acc : {test_acc:.2f}%  |  AUC: {auc:.4f}")

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"  ✓ Best model saved (acc={best_acc:.2f}%)")

    scheduler.step()


Epoch 1/20 [eval] : 100%|██████████| 21/21 [00:02<00:00,  9.45it/s]



Epoch 1/20
  Train Acc : 81.59%
  Test  Acc : 81.18%  |  AUC: 0.9017
  ✓ Best model saved (acc=81.18%)


Epoch 2/20 [eval] : 100%|██████████| 21/21 [00:02<00:00,  9.79it/s]



Epoch 2/20
  Train Acc : 92.58%
  Test  Acc : 86.71%  |  AUC: 0.9629
  ✓ Best model saved (acc=86.71%)


Epoch 3/20 [eval] : 100%|██████████| 21/21 [00:02<00:00,  9.77it/s]


...


Epoch 18/20
  Train Acc : 99.50%
  Test  Acc : 92.24%  |  AUC: 0.9749


Epoch 19/20 [eval] : 100%|██████████| 21/21 [00:02<00:00,  9.23it/s]



Epoch 19/20
  Train Acc : 99.38%
  Test  Acc : 91.40%  |  AUC: 0.9746


Epoch 20/20 [eval] : 100%|██████████| 21/21 [00:02<00:00,  9.74it/s]


Epoch 20/20
  Train Acc : 99.30%
  Test  Acc : 91.09%  |  AUC: 0.9739


In [22]:
import torch
import onnx

# Load best checkpoint
model_export = MobileNetTexture().to(device)
model_export.load_state_dict(
    torch.load(CKPT_PATH, map_location=device)
)
model_export.eval()

print("✓ Best model loaded for export")

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


✓ Best model loaded for export


In [23]:
# Fixed single-frame input
dummy_input = torch.randn(1, 3, 224, 224).to(device)

onnx_path = "spoofnet.onnx"

torch.onnx.export(
    model_export,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes=None,                    # ← remove dynamic axes
    external_data=False        # ← FORCE single file
)

print(f"✓ ONNX exported → {onnx_path}")

W0220 10:25:16.755000 2424 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0220 10:25:16.757000 2424 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0220 10:25:16.758000 2424 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0220 10:25:16.760000 2424 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `MobileNetTexture([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetTexture([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 78 of general pattern rewrite rules.
✓ ONNX exported → spoofnet.onnx
